# SmartVision - Step 1: Test ImageNet ResNet-50 on your own images

**What this notebook does (in simple words)**

1. Loads ResNet-50 that has *already learned* from 1.2 million real photos (ImageNet). No training needed.
2. You upload 20-30 random images (for example from Google).
3. The model predicts the top-3 labels for each image and tells you how confident it is.
4. If confidence is low, it says **"Not sure"** instead of guessing wrongly.
5. You count how many predictions were right. That number decides your next step (Route A or Route B).

**Before you run:** turn on the GPU.
Colab: `Runtime > Change runtime type > T4 GPU`. Kaggle: `Settings > Accelerator > GPU T4`.

## 1. Setup

In [ ]:
import os, time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NOT FOUND - enable a GPU in the runtime settings (it still works on CPU, just slower)")

## 2. Load the pretrained ResNet-50

`weights="imagenet"` downloads the already-trained knowledge (about 100 MB, only once).
This model can name **1000 everyday things** (dogs, cats, cars, food, objects and more).

In [ ]:
t0 = time.time()
model = ResNet50(weights="imagenet")
print(f"Model ready in {time.time() - t0:.1f} s  |  parameters: {model.count_params():,}")

## 3. Get your test images

Use **20-30 different images** from Google. Mix easy ones (a clear dog, a car) with hard ones
(busy background, unusual angle, something that is probably NOT in the 1000 classes).

- **Colab:** run the cell below and pick your images.
- **Kaggle:** add your images as a Dataset, then set `IMAGE_DIR` to its path, for example
  `IMAGE_DIR = "/kaggle/input/my-test-images"`.

In [ ]:
IMAGE_DIR = "my_images"          # Kaggle users: change this to your dataset path
os.makedirs(IMAGE_DIR, exist_ok=True)

try:
    from google.colab import files       # only exists on Colab
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(IMAGE_DIR, name), "wb") as f:
            f.write(data)
    print(f"Saved {len(uploaded)} image(s) to '{IMAGE_DIR}'")
except ImportError:
    print(f"Not on Colab - using images already inside '{IMAGE_DIR}'")

VALID_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
image_paths = sorted(
    os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.lower().endswith(VALID_EXT)
)
print("Images found:", len(image_paths))

## 4. Prepare images the right way

This is where many projects go wrong, so read carefully.

- Fix phone-rotation (EXIF), convert to RGB.
- **Resize and centre-crop to 224x224** (this keeps the shape natural and does not squash the image).
- Use ResNet's own `preprocess_input`. The model was trained with exactly this maths,
  so using anything else (for example dividing by 255) gives bad predictions.

Later, in your real project, **the same function must be used in training and in FastAPI**.

In [ ]:
def load_and_prepare(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    img = ImageOps.fit(img, (224, 224), Image.BILINEAR)      # resize + centre crop
    arr = np.asarray(img, dtype="float32")
    return img, preprocess_input(arr.copy())

## 5. Predict (all images in one go)

We send all images through the model together in a batch. This is much faster than one by one,
and the timing shown will help you with your speed worries.

**Confidence threshold:** if the top prediction is below `CONF_THRESHOLD`, the answer becomes
*"Not sure"*. This fixes the problem of the model being forced to pick something for unknown images.

In [ ]:
CONF_THRESHOLD = 0.40     # 40 percent - you can tune this later

shown_images, batch = [], []
for p in image_paths:
    try:
        img, x = load_and_prepare(p)
        shown_images.append(img)
        batch.append(x)
    except Exception as e:
        print("Skipped (could not open):", p, "->", e)

batch = np.stack(batch)
print("Batch shape:", batch.shape)

t0 = time.time()
preds = model.predict(batch, batch_size=32, verbose=0)
elapsed = time.time() - t0
print(f"Predicted {len(batch)} images in {elapsed:.2f} s  ({1000 * elapsed / len(batch):.0f} ms per image)")

decoded = decode_predictions(preds, top=3)      # list of [(id, label, prob), ...]

## 6. See the results

In [ ]:
rows = []
for path, top3 in zip(image_paths, decoded):
    (_, l1, p1), (_, l2, p2), (_, l3, p3) = top3
    rows.append({
        "file": os.path.basename(path),
        "top1": l1.replace("_", " "), "conf1": round(float(p1), 3),
        "top2": l2.replace("_", " "), "conf2": round(float(p2), 3),
        "top3": l3.replace("_", " "), "conf3": round(float(p3), 3),
        "status": "Confident" if p1 >= CONF_THRESHOLD else "Not sure",
    })

results = pd.DataFrame(rows)
results.to_csv("step1_results.csv", index=False)
results

In [ ]:
cols = 4
rows_n = int(np.ceil(len(shown_images) / cols))
fig, axes = plt.subplots(rows_n, cols, figsize=(4 * cols, 4.2 * rows_n))
axes = np.array(axes).reshape(-1)

for ax, img, r in zip(axes, shown_images, rows):
    ax.imshow(img)
    ax.axis("off")
    if r["status"] == "Confident":
        title = f"{r['top1']}  ({r['conf1']:.0%})"
        color = "green"
    else:
        title = f"Not sure\n(best guess: {r['top1']}, {r['conf1']:.0%})"
        color = "darkorange"
    ax.set_title(title, fontsize=10, color=color)

for ax in axes[len(shown_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Score it yourself (this is your decision maker)

Look at the pictures above and count honestly.

- `TOP1_CORRECT` = how many images where the **first** label was right
- `TOP3_CORRECT` = how many images where the right answer was **anywhere in the top 3**
- `IN_SCOPE` = how many of your images show something that really is an everyday object/animal/food
  (skip weird ones such as logos, screenshots, or very rare things)

Edit the three numbers, then run the cell.

In [ ]:
TOP1_CORRECT = 0     # <- edit
TOP3_CORRECT = 0     # <- edit
IN_SCOPE     = len(image_paths)   # <- edit if some images were not everyday things

if IN_SCOPE > 0:
    print(f"Top-1 accuracy on YOUR images: {TOP1_CORRECT / IN_SCOPE:.0%}")
    print(f"Top-3 accuracy on YOUR images: {TOP3_CORRECT / IN_SCOPE:.0%}")
    print(f"'Not sure' answers: {(results['status'] == 'Not sure').sum()} of {len(results)}")

    top1 = TOP1_CORRECT / IN_SCOPE
    if top1 >= 0.75:
        print("\n-> Route A works well. Use ImageNet ResNet-50 as your base and fine-tune only if you need custom classes.")
    elif top1 >= 0.5:
        print("\n-> Decent. Fine-tune the head on a real-photo dataset (Route B) to improve it.")
    else:
        print("\n-> Weak. Check preprocessing first, then choose a dataset closer to your images (Route B).")

## What the results mean

| Result | What it tells you |
|---|---|
| Most images correct and confident | ImageNet ResNet-50 already handles real-world photos. Your project can work today. |
| Many "Not sure" on things you expected to work | The object may not be one of the 1000 classes. Fine-tune with your own categories (Route B). |
| Wrong but very confident answers | Look at the image. It may look like a similar class, or the photo may be unusual. |

**Next step:** save `step1_results.csv`, note your Top-1 and Top-3 numbers, and come back to decide
Route A or Route B.